# RG3 gas case matching and historical trajectories
Descriptive case analysis. Future normal references and whole-block scales are never used for predictive claims.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables';figdir=root/'reports/figures/rg3_gas_cases'
figdir.mkdir(parents=True,exist_ok=True)
plt.rcParams['font.family']='Malgun Gothic'
plt.rcParams['axes.unicode_minus']=False
old=json.loads((root/'notebooks/10_original_defect_report_revision.ipynb').read_text(encoding='utf-8'))
prep=''.join(next(c for c in old['cells'] if c['cell_type']=='code')['source'])
exec(prep.split('for feature in features:')[0])
shots=label_shots.merge(observed,on=keys,how='left',validate='one_to_one')
rg=shots.loc[shots['product'].eq('RG3') & (shots.PassOrFail.eq(0)|shots.Reason.str.contains('가스',regex=False))].reset_index(drop=True)
rg['gas']=rg.Reason.str.contains('가스',regex=False)
assert len(rg)==583 and rg.gas.sum()==17
chosen=['Average_Back_Pressure','Max_Injection_Pressure','Cushion_Position','Hopper_Temperature','Barrel_Temperature_5','Mold_Temperature_4']
names={'Average_Back_Pressure':'평균 배압','Max_Injection_Pressure':'최대 사출압력','Cushion_Position':'쿠션 위치','Hopper_Temperature':'호퍼 온도','Barrel_Temperature_5':'배럴5 온도','Mold_Temperature_4':'금형4 온도'}
# Historical trend excludes target and uses prior 5 versus preceding 15 observed shots.
for f in chosen:
    prev=observed.groupby('history_segment')[f].shift()
    last5=prev.groupby(observed.history_segment).transform(lambda s:s.rolling(5,min_periods=5).mean())
    earlier=prev.groupby(observed.history_segment).shift(5)
    earlier15=earlier.groupby(observed.history_segment).transform(lambda s:s.rolling(15,min_periods=15).mean())
    observed[f+'__past_trend']=last5-earlier15
trendcols=[f+'__past_trend' for f in chosen]
rg=rg.merge(observed[keys+trendcols],on=keys,how='left',validate='one_to_one')
# Same-shot side audit, using original label rows rather than assumed label absence.
rawrg=labels.loc[labels['product'].eq('RG3')]
side_rows=[]
for _,shot in rg.loc[rg.gas].iterrows():
    sides=rawrg.loc[rawrg.TimeStamp.eq(shot.TimeStamp)&rawrg.EQUIP_CD.eq(shot.EQUIP_CD)]
    lhs=sides.loc[sides.side.eq('LH')];rhs=sides.loc[sides.side.eq('RH')]
    side_rows.append(dict(TimeStamp=shot.TimeStamp,LH_present=len(lhs)>0,LH_normal=bool(len(lhs)>0 and lhs.PassOrFail.eq(0).all()),RH_gas=bool(len(rhs)>0 and rhs.Reason.str.contains('가스',regex=False).any()),max_sensor_side_difference=float((sides[features].max()-sides[features].min()).abs().max())))
side=pd.DataFrame(side_rows)
assert len(side)==17 and side.LH_normal.all() and side.RH_gas.all() and side.max_sensor_side_difference.le(1e-4).all()
matchrows=[];summary=[];distribution=[];trend_distribution=[];normal_reference=[]
for block,g in rg.groupby('comparison_block'):
    normal=g.loc[~g.gas].copy();bad=g.loc[g.gas].copy()
    # Descriptive only: all labeled normals in this block define scales, may include future.
    center=normal[features].median();scale=normal[features].std(ddof=0)
    usable=scale.index[scale.gt(1e-8)&scale.notna()].tolist()
    zn=(normal[usable]-center[usable])/scale[usable]
    neighbors=NearestNeighbors(n_neighbors=2).fit(zn)
    normal_d=neighbors.kneighbors(zn)[0][:,1]/np.sqrt(len(usable))
    for idx,value in zip(normal.index,normal_d):normal_reference.append(dict(block=block,TimeStamp=normal.loc[idx,'TimeStamp'],nearest_other_normal_distance=float(value)))
    for f in features:
        n=normal[f].dropna();b=bad[f].dropna();q10,q90=n.quantile(.1),n.quantile(.9)
        distribution.append(dict(block=block,feature=f,normal_n=len(n),gas_n=len(b),normal_median=n.median(),gas_median=b.median(),median_difference=b.median()-n.median(),gas_outside_middle80=int(((b<q10)|(b>q90)).sum())))
    for f in trendcols:
        n=normal[f].dropna();b=bad[f].dropna();q10,q90=n.quantile(.1),n.quantile(.9)
        trend_distribution.append(dict(block=block,feature=f,normal_n=len(n),gas_n=len(b),normal_median=n.median(),gas_median=b.median(),median_difference=b.median()-n.median(),normal_q10=q10,normal_q90=q90,gas_outside_middle80=int(((b<q10)|(b>q90)).sum())))
    for idx,shot in bad.iterrows():
        z=((shot[usable]-center[usable])/scale[usable]).astype(float).to_numpy()
        sd,ni=neighbors.kneighbors(z.reshape(1,-1),n_neighbors=1)
        sens=normal.iloc[int(ni[0,0])]
        d=float(sd[0,0]/np.sqrt(len(usable)))
        timeidx=(normal.TimeStamp-shot.TimeStamp).abs().idxmin()
        time=normal.loc[timeidx]
        before=normal.loc[normal.TimeStamp.lt(shot.TimeStamp)].sort_values('TimeStamp').tail(1)
        after=normal.loc[normal.TimeStamp.gt(shot.TimeStamp)].sort_values('TimeStamp').head(1)
        candidates=[('nearest_time',time),('nearest_sensor',sens)]
        if len(before):candidates.append(('previous_normal',before.iloc[0]))
        if len(after):candidates.append(('next_normal',after.iloc[0]))
        for kind,ref in candidates:
            for f in features:
                matchrows.append(dict(gas_time=shot.TimeStamp,block=block,match_type=kind,normal_time=ref.TimeStamp,minutes_difference=(ref.TimeStamp-shot.TimeStamp).total_seconds()/60,feature=f,gas_value=shot[f],normal_value=ref[f],difference=shot[f]-ref[f],difference_in_block_normal_sd=(shot[f]-ref[f])/scale[f] if scale[f]>1e-8 else np.nan))
        contrast=((shot[usable]-time[usable])/scale[usable]).astype(float)
        top=contrast.abs().sort_values(ascending=False).head(3).index.tolist()
        ranked=[f'{f}: {shot[f]:.4f} / 정상 {time[f]:.4f} (차이 {contrast[f]:+.2f}σ)' for f in top]
        gastrend=[f for f in trendcols if pd.notna(shot[f])]
        count=0
        for f in gastrend:
            n=normal[f].dropna()
            count+=int(shot[f]<n.quantile(.1) or shot[f]>n.quantile(.9))
        summary.append(dict(TimeStamp=shot.TimeStamp,comparison_block=block,nearest_time_normal=time.TimeStamp,time_gap_seconds=abs((time.TimeStamp-shot.TimeStamp).total_seconds()),previous_normal_time=before.iloc[0].TimeStamp if len(before) else pd.NaT,next_normal_time=after.iloc[0].TimeStamp if len(after) else pd.NaT,nearest_sensor_normal=sens.TimeStamp,sensor_distance=d,distance_normal_reference_percentile=float((normal_d<=d).mean()*100),matching_feature_count=len(usable),time_match_top_differences='; '.join(ranked),past_trend_available=len(gastrend),past_trend_outside_normal_middle80=count))
summary=pd.DataFrame(summary).sort_values('TimeStamp').reset_index(drop=True)
# Plot +/-15 minutes only within the same history segment. All unlabeled states remain unknown.
annot=label_shots[keys+['PassOrFail','Reason']]
plotobs=observed.merge(annot,on=keys,how='left',validate='one_to_one')
for case,row in summary.iterrows():
    shot=rg.loc[rg.TimeStamp.eq(row.TimeStamp)].iloc[0]
    lo=row.TimeStamp-pd.Timedelta(minutes=15);hi=row.TimeStamp+pd.Timedelta(minutes=15)
    g=plotobs.loc[plotobs.history_segment.eq(shot.history_segment)&plotobs.TimeStamp.between(lo,hi)].sort_values('TimeStamp')
    x=(g.TimeStamp-row.TimeStamp).dt.total_seconds()/60
    # Break lines at gaps >3min rather than imply continuous sensor readings.
    breakmask=g.TimeStamp.diff().dt.total_seconds().gt(180)
    fig,axes=plt.subplots(2,3,figsize=(14,7))
    for ax,f in zip(axes.flat,chosen):
        values=g[f].copy();values.loc[breakmask]=np.nan
        ax.plot(x,values,color='#9aa0a6',lw=.9,label='관측 이력(비라벨 포함)')
        mask=g.PassOrFail.eq(0)
        ax.scatter(x[mask],g.loc[mask,f],color='#1b9e77',s=17,label='라벨 정상',zorder=3)
        gasmask=g.Reason.fillna('').str.contains('가스',regex=False)
        ax.scatter(x[gasmask],g.loc[gasmask,f],color='#d62728',s=40,marker='x',label='가스',zorder=4)
        ax.axvline(0,color='#d62728',lw=1,ls='--')
        time=rg.loc[rg.TimeStamp.eq(row.nearest_time_normal)].iloc[0]
        xt=(time.TimeStamp-row.TimeStamp).total_seconds()/60
        if abs(xt)<=15:ax.scatter([xt],[time[f]],s=90,facecolors='none',edgecolors='#1b9e77',linewidths=1.8,label='가까운 정상',zorder=5)
        ax.set_title(names[f]);ax.set_xlabel('가스 발생 기준 분');ax.set_xlim(-15,15);ax.grid(alpha=.2);ax.ticklabel_format(axis='y',style='plain',useOffset=False)
    handles,lab=axes.flat[0].get_legend_handles_labels()
    fig.legend(handles,lab,loc='lower center',ncol=4,fontsize=9)
    fig.suptitle(f'RG3 가스 {case+1:02d} — {row.TimeStamp} / RH 가스·LH 정상',fontsize=14)
    fig.tight_layout(rect=(0,.06,1,.95))
    fig.savefig(figdir/f'case_{case+1:02d}.png',dpi=140)
    plt.close(fig)
summary.to_csv(tables/'rg3_gas_matched_case_summary.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(matchrows).to_csv(tables/'rg3_gas_normal_sensor_comparisons.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(distribution).to_csv(tables/'rg3_gas_raw_feature_overlap.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(trend_distribution).to_csv(tables/'rg3_gas_past_trend_overlap.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(normal_reference).to_csv(tables/'rg3_normal_nearest_neighbor_reference.csv',index=False,encoding='utf-8-sig')
side.to_csv(tables/'rg3_gas_side_sensor_audit.csv',index=False,encoding='utf-8-sig')
assert len(summary)==17 and len(list(figdir.glob('case_*.png')))==17
rawstats=pd.DataFrame(distribution).groupby('feature').agg(gas_available=('gas_n','sum'),outside=('gas_outside_middle80','sum'))
trendstats=pd.DataFrame(trend_distribution).groupby('feature').agg(gas_available=('gas_n','sum'),outside=('gas_outside_middle80','sum'))
lines=['# RG3 가스17개: 정상과일대일비교 및 전후흐름','',
'원본라벨의같은설비·제품·Timestamp를합친Shot기준. 정상566개와가스17개. 미성형8개제외. 이보고서는사례설명이며 새모델성능평가가아니다.',
'','## 비교방법','',
'- 시간비교: 같은라벨관측구간에서가장가까운정상,직전정상,직후정상. 이후정상은설명용으로만사용한다.',
'- 센서비교: 같은구간정상표준편차로표준화한가변센서의평균제곱거리로가장유사한정상선정. 상수센서제외. 전체센서로이웃을고르므로유사성이높은것은선정방식의결과이기도하다.',
'- 거리백분위: 같은구간정상끼리자기자신을제외한최근접거리분포에서해당가스의거리위치. 낮을수록정상끼리의거리와비교해도정상과가깝다. 판정확률이나검증된임계값이아니다.',
'- 같은구간전체정상으로범위·표준편차를계산하므로미래정보도포함한다. 전부설명용이고모델입력생성이나성능주장에사용하지않았다.',
'- 전후그래프는동일연속관측이력내±15분이며비라벨은정상으로간주하지않는다. 3분초과공백은선으로연결하지않았다. 이력구간은실제Run ID가아니다.',
'- 발생전추세는현재를제외한직전5개평균−그이전15개평균.20개모두있어야계산한다.',
'','## 핵심 결과','',
'가스17개모두60~62초간격의정상과비교가능했다. 센서최근접거리의정상참조90백분위안에14/17개,95백분위안에16/17개가들었다. 거리규칙을학습·검증한결과가아니라정상과의겹침설명이다.',
'','발생전20개이력은16/17개에서확보됐다. 10/23 01:03:33은이력이부족해추세판정하지않는다. 추세0개라는표시는유효이력이있을때만변화없음의비교에사용한다.',
'','가까운정상과한센서가크게다르더라도다른정상에는유사한값이있을수있다. 24개중큰차이3개를사후선택했으므로큰차이가존재한다는것만으로가스신호라고주장하지않는다. 현재그래프와분포에서가스직전공통된온도상승또는쿠션급변규칙은확인되지않았다.',
'','## RH/LH 확인','',
'가스17개모두RH가스이고같은Timestamp의LH는정상라벨이다. 분석센서24개는각Shot에서좌우가동일하다(최대차이<=0.0001). 현재센서는좌우캐비티별차이를직접보여주지않는다. 이사실만으로Shot단위가스판별불가능을뜻하지않는다.',
'','## 모든가스와정상비교','',
'거리백분위는작을수록정상과유사하며,큰값도다중센서이상일뿐가스원인확정은아니다.',
'','| 번호 | 가스시각 | 가까운정상시각 | 간격초 | 센서유사정상시각 | 거리백분위 | 발생전추세확보/6 | 정상80%범위밖추세수 |',
'|---|---|---|---:|---|---:|---:|---:|']
for idx,r in summary.iterrows():lines.append(f'| {idx+1} | {r.TimeStamp} | {r.nearest_time_normal} | {r.time_gap_seconds:.0f} | {r.nearest_sensor_normal} | {r.distance_normal_reference_percentile:.1f}% | {r.past_trend_available} | {r.past_trend_outside_normal_middle80} |')
lines+=['','## 발생전추세','',
'같은구간정상의가운데80%범위밖에있는가스수. 정상도약20%가범위밖에있도록만든설명용범위이며,여러변수를동시에비교했으므로단순발견수만으로유의성을주장하지않는다.',
'','| 센서 | 추세계산가능가스 | 범위밖가스 |','|---|---:|---:|']
for f in chosen:
    r=trendstats.loc[f+'__past_trend'];lines.append(f'| {names[f]} | {int(r.gas_available)} | {int(r.outside)} |')
lines+=['','## 사례별전후그래프','',
'주변가스도빨간X로표시했다. 비라벨점의품질은알수없다. 정상비교의큰차이3개는설명용사후선택이며예측특징으로확정하지않는다.']
for idx,r in summary.iterrows():
    lines+=['',f'### {idx+1:02d}. {r.TimeStamp}','',
    '가까운정상과의차이: '+r.time_match_top_differences.replace('正常','정상').replace('差','차이'),
    '',f'![가스사례{idx+1:02d}](figures/rg3_gas_cases/case_{idx+1:02d}.png)']
(root/'reports/rg3_gas_matched_case_review.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('SUMMARY\n'+summary[['TimeStamp','time_gap_seconds','sensor_distance','distance_normal_reference_percentile','past_trend_available','past_trend_outside_normal_middle80']].to_string(index=False))
print('PAST TREND\n'+trendstats.to_string())
print('RAW OVERLAP TOP\n'+rawstats.sort_values('outside',ascending=False).head(10).to_string())
print('Verified17 cases,17 figures and side labels.',flush=True)



Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\hayeon\\.matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but NearestNe

C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but NearestNe

SUMMARY
          TimeStamp  time_gap_seconds  sensor_distance  distance_normal_reference_percentile  past_trend_available  past_trend_outside_normal_middle80
2020-10-22 00:51:52              60.0         0.960560                             87.755102                     6                                   2
2020-10-22 01:04:15              61.0         0.726316                             26.530612                     6                                   1
2020-10-22 01:26:54              62.0         0.819053                             57.142857                     6                                   0
2020-10-22 01:57:47              62.0         0.815998                             57.142857                     6                                   0
2020-10-22 04:46:36              62.0         0.829002                             77.540107                     6                                   3
2020-10-22 04:51:46              60.0         0.875615                             85.